# CVRP Cluster-First Sweep, Route-Second TSP QUBOs

This notebook decomposes a CVRP instance with a classical sweep clustering step, then converts each fixed cluster into a route-second TSP QUBO. It stops at QUBO conversion. It does not run VQE, QAOA, sampling, or any quantum simulation.


## Why this uses fewer qubits

The full arc-MTZ CVRP encoding couples all customers, all route arcs, capacity variables, and MTZ slack variables into one large QUBO. For this instance that estimate was 4320 binary variables.

Sweep clustering first fixes which customers are considered together. The quantum part only sees one TSP route at a time. A route with `r` customers uses `r * r` binary variables in this assignment encoding. For the default instance, the largest sweep cluster has 6 customers, so the sequential route-second QUBO width is `6 * 6 = 36` qubits. If every route QUBO were combined into one independent block model, the total would be 86 variables.

Tradeoff: the greedy sweep shown here creates 5 capacity-feasible clusters, while the file name indicates `k=4`. This is a decomposition heuristic for reducing qubits; it is not preserving the original route-count constraint in this basic form.


In [1]:
from pathlib import Path
import math
import re

import numpy as np
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo


def find_cvrp_dir():
    candidates = [
        Path('cvrp'),
        Path('individual') / 'cvrp',
        Path.cwd() / 'cvrp',
        Path.cwd() / 'individual' / 'cvrp',
    ]
    for candidate in candidates:
        if candidate.exists() and any(candidate.glob('*.vrp')):
            return candidate
    raise FileNotFoundError('Could not find a cvrp folder containing .vrp files.')


def read_cvrp_instance(file_path):
    file_path = Path(file_path)
    metadata = {}
    coords = {}
    demands = {}
    depots = []
    section = None

    for raw_line in file_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if line == 'EOF':
            break
        if line in {'NODE_COORD_SECTION', 'DEMAND_SECTION', 'DEPOT_SECTION'}:
            section = line
            continue

        if section == 'NODE_COORD_SECTION':
            node, x_coord, y_coord = line.split()[:3]
            coords[int(node)] = (float(x_coord), float(y_coord))
            continue

        if section == 'DEMAND_SECTION':
            node, demand = line.split()[:2]
            demands[int(node)] = int(float(demand))
            continue

        if section == 'DEPOT_SECTION':
            node = int(line.split()[0])
            if node != -1:
                depots.append(node)
            continue

        if ':' in line:
            key, value = line.split(':', 1)
            metadata[key.strip()] = value.strip().strip('"')

    if not depots:
        raise ValueError(f'No depot found in {file_path}')
    if set(coords) != set(demands):
        raise ValueError('Coordinate and demand sections have different node ids.')

    depot = depots[0]
    nodes = sorted(coords)
    customers = [node for node in nodes if node != depot]
    return {
        'path': file_path,
        'metadata': metadata,
        'coords': coords,
        'demands': demands,
        'depots': depots,
        'depot': depot,
        'nodes': nodes,
        'customers': customers,
    }


def parse_vehicle_count(instance, default=1):
    name = instance['metadata'].get('NAME', instance['path'].stem)
    match = re.search(r'-k(\d+)', name)
    return int(match.group(1)) if match else default


def euclidean_distance(instance, a, b):
    ax, ay = instance['coords'][a]
    bx, by = instance['coords'][b]
    return int(round(math.hypot(ax - bx, ay - by)))


def bit_width_from_qubo(qubo):
    return qubo.get_num_vars()


def print_instance_summary(instance):
    capacity = int(instance['metadata']['CAPACITY'])
    parsed_vehicles = parse_vehicle_count(instance, default=len(instance['customers']))
    total_demand = sum(instance['demands'][node] for node in instance['customers'])
    print(f"Instance: {instance['metadata'].get('NAME', instance['path'].stem)}")
    print(f"Comment: {instance['metadata'].get('COMMENT', '')}")
    print(f"Customers: {len(instance['customers'])}")
    print(f"Depot: {instance['depot']}")
    print(f"Parsed vehicles: {parsed_vehicles}")
    print(f"Capacity: {capacity}")
    print(f"Total demand: {total_demand}")
    print(f"Demand / capacity: {total_demand / capacity:.2f}")


In [2]:
INSTANCE_NAME = 'XSH-n20-k4-01.vrp'
CVRP_DIR = find_cvrp_dir()
instance = read_cvrp_instance(CVRP_DIR / INSTANCE_NAME)
parsed_vehicle_count = parse_vehicle_count(instance, default=len(instance['customers']))
vehicle_capacity = int(instance['metadata']['CAPACITY'])

print(f"CVRP directory: {CVRP_DIR.resolve()}")
print_instance_summary(instance)


CVRP directory: /Users/monitsharma/SMU-Quantum-Repos/autoqresearch/individual/cvrp
Instance: XSH-n20-k4-01
Comment: Queiroga and Uchoa (2024); Optimal cost: 646
Customers: 20
Depot: 1
Parsed vehicles: 4
Capacity: 231
Total demand: 924
Demand / capacity: 4.00


In [3]:
def route_second_tsp_assignment_qubits(clusters):
    per_cluster = [len(cluster) ** 2 for cluster in clusters]
    return {
        'per_cluster': per_cluster,
        'sequential_qubits': max(per_cluster) if per_cluster else 0,
        'combined_qubits': sum(per_cluster),
    }


def cluster_summary(clusters, demands):
    return [
        {
            'cluster': index,
            'size': len(cluster),
            'load': sum(demands[customer] for customer in cluster),
            'customers': cluster,
        }
        for index, cluster in enumerate(clusters)
    ]


def print_cluster_summary(clusters, instance):
    for row in cluster_summary(clusters, instance['demands']):
        print(
            f"cluster={row['cluster']} size={row['size']} "
            f"load={row['load']} customers={row['customers']}"
        )
    route_qubits = route_second_tsp_assignment_qubits(clusters)
    print()
    print(f"Sequential route-second qubits: {route_qubits['sequential_qubits']}")
    print(f"Combined route-second qubits: {route_qubits['combined_qubits']}")
    print(f"Per-route qubits: {route_qubits['per_cluster']}")


def build_route_second_tsp_qp(instance, cluster, name):
    """Build a TSP assignment model for one fixed CVRP cluster.

    Variables y_customer_position mean customer is visited at that route position.
    The depot is fixed outside the assignment: depot -> first customer -> ... -> last customer -> depot.
    This model does not decide which customers belong to the cluster; that was fixed by the cluster-first step.
    """
    depot = instance['depot']
    n = len(cluster)
    qp = QuadraticProgram(name=name)

    if n == 0:
        raise ValueError('Cannot build a route model for an empty cluster.')

    for customer in cluster:
        for position in range(n):
            qp.binary_var(name=f'y_{customer}_{position}')

    linear = {}
    quadratic = {}

    # Cost from depot to first customer and last customer back to depot.
    for customer in cluster:
        linear[f'y_{customer}_0'] = linear.get(f'y_{customer}_0', 0.0) + euclidean_distance(instance, depot, customer)
        linear[f'y_{customer}_{n - 1}'] = linear.get(f'y_{customer}_{n - 1}', 0.0) + euclidean_distance(instance, customer, depot)

    # Cost between consecutive route positions.
    for position in range(n - 1):
        for i in cluster:
            for j in cluster:
                if i == j:
                    continue
                key = (f'y_{i}_{position}', f'y_{j}_{position + 1}')
                quadratic[key] = quadratic.get(key, 0.0) + euclidean_distance(instance, i, j)

    qp.minimize(linear=linear, quadratic=quadratic)

    for customer in cluster:
        qp.linear_constraint(
            linear={f'y_{customer}_{position}': 1 for position in range(n)},
            sense='==',
            rhs=1,
            name=f'visit_customer_{customer}',
        )

    for position in range(n):
        qp.linear_constraint(
            linear={f'y_{customer}_{position}': 1 for customer in cluster},
            sense='==',
            rhs=1,
            name=f'one_customer_at_position_{position}',
        )

    return qp


def convert_cluster_tsp_models_to_qubo(instance, clusters, prefix):
    route_models = []
    for index, cluster in enumerate(clusters):
        qp = build_route_second_tsp_qp(instance, cluster, name=f'{prefix}_route_{index}')
        converter = QuadraticProgramToQubo()
        qubo = converter.convert(qp)
        route_models.append(
            {
                'cluster_index': index,
                'customers': cluster,
                'load': sum(instance['demands'][customer] for customer in cluster),
                'qp': qp,
                'qubo': qubo,
                'converter_penalty': converter.penalty,
            }
        )
    return route_models


def print_route_qubo_summary(route_models):
    print('| route | customers | load | QP vars | QP constraints | QUBO vars | penalty |')
    print('| --- | --- | --- | --- | --- | --- | --- |')
    for route in route_models:
        qp = route['qp']
        qubo = route['qubo']
        print(
            f"| {route['cluster_index']} | {route['customers']} | {route['load']} | "
            f"{qp.get_num_vars()} | {len(qp.linear_constraints)} | "
            f"{qubo.get_num_vars()} | {route['converter_penalty']} |"
        )


def preview_largest_route_qubo(route_models, max_chars=3500):
    largest_route = max(route_models, key=lambda row: row['qubo'].get_num_vars())
    qubo_lp = largest_route['qubo'].export_as_lp_string()
    print(f"Largest route index: {largest_route['cluster_index']}")
    print(f"Largest route QUBO variables: {largest_route['qubo'].get_num_vars()}")
    print(qubo_lp[:max_chars])
    if len(qubo_lp) > max_chars:
        print()
        print(f"... truncated {len(qubo_lp) - max_chars} characters")


## Sweep clustering

Customers are sorted by polar angle around the depot. The algorithm scans them in that angular order and starts a new cluster when adding the next demand would exceed vehicle capacity. This is a classical preprocessing step; no optimization solver is called.


In [4]:
def sweep_clusters(instance):
    depot = instance['depot']
    depot_x, depot_y = instance['coords'][depot]
    ordered = sorted(
        instance['customers'],
        key=lambda customer: math.atan2(
            instance['coords'][customer][1] - depot_y,
            instance['coords'][customer][0] - depot_x,
        ),
    )
    clusters = []
    current_cluster = []
    current_load = 0
    capacity = int(instance['metadata']['CAPACITY'])

    for customer in ordered:
        demand = instance['demands'][customer]
        if current_cluster and current_load + demand > capacity:
            clusters.append(current_cluster)
            current_cluster = []
            current_load = 0
        current_cluster.append(customer)
        current_load += demand

    if current_cluster:
        clusters.append(current_cluster)
    return clusters


sweep_cluster_list = sweep_clusters(instance)
print_cluster_summary(sweep_cluster_list, instance)
print(f"Parsed vehicle count: {parsed_vehicle_count}")


cluster=0 size=4 load=145 customers=[16, 10, 3, 4]
cluster=1 size=3 load=189 customers=[7, 19, 20]
cluster=2 size=6 load=218 customers=[8, 6, 2, 9, 21, 12]
cluster=3 size=3 load=173 customers=[5, 18, 11]
cluster=4 size=4 load=199 customers=[13, 14, 17, 15]

Sequential route-second qubits: 36
Combined route-second qubits: 86
Per-route qubits: [16, 9, 36, 9, 16]
Parsed vehicle count: 4


## Route-second TSP QUBO construction

For each fixed cluster, the QUBO uses `y_customer_position` variables. Each customer must appear in exactly one route position, and each route position must contain exactly one customer. The depot is fixed at the start and end, so depot arcs are linear objective terms and customer-to-customer transitions are quadratic terms.


In [5]:
sweep_route_models = convert_cluster_tsp_models_to_qubo(
    instance,
    sweep_cluster_list,
    prefix='sweep_tsp',
)
print_route_qubo_summary(sweep_route_models)

sweep_qubits = route_second_tsp_assignment_qubits(sweep_cluster_list)
print()
print(f"Sequential qubits needed: {sweep_qubits['sequential_qubits']}")
print(f"Combined independent route blocks: {sweep_qubits['combined_qubits']}")


| route | customers | load | QP vars | QP constraints | QUBO vars | penalty |
| --- | --- | --- | --- | --- | --- | --- |
| 0 | [16, 10, 3, 4] | 145 | 16 | 8 | 16 | 947.0 |
| 1 | [7, 19, 20] | 189 | 9 | 6 | 9 | 687.0 |
| 2 | [8, 6, 2, 9, 21, 12] | 218 | 36 | 12 | 36 | 3791.0 |
| 3 | [5, 18, 11] | 173 | 9 | 6 | 9 | 341.0 |
| 4 | [13, 14, 17, 15] | 199 | 16 | 8 | 16 | 971.0 |

Sequential qubits needed: 36
Combined independent route blocks: 86


## Largest route QUBO preview

The following cell prints only a preview of the largest route QUBO LP string. This is still classical QUBO conversion output, not a quantum simulation.


In [6]:
preview_largest_route_qubo(sweep_route_models)


/var/folders/tm/6bh1bn3x6pgfgp8nvpknylq40000gn/T/ipykernel_37149/2966372286.py:125: DeprecationWarning: The method ``qiskit_optimization.problems.quadratic_program.QuadraticProgram.export_as_lp_string()`` is deprecated as of Qiskit 0.7.0. It will be removed no earlier than 3 months after the release date. Use prettyprint instead.
  qubo_lp = largest_route['qubo'].export_as_lp_string()


Largest route index: 2
Largest route QUBO variables: 36
\ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: sweep_tsp_route_2

Minimize
 obj: - 15096 y_8_0 - 15164 y_8_1 - 15164 y_8_2 - 15164 y_8_3 - 15164 y_8_4
      - 15096 y_8_5 - 15113 y_6_0 - 15164 y_6_1 - 15164 y_6_2 - 15164 y_6_3
      - 15164 y_6_4 - 15113 y_6_5 - 15116 y_2_0 - 15164 y_2_1 - 15164 y_2_2
      - 15164 y_2_3 - 15164 y_2_4 - 15116 y_2_5 - 15126 y_9_0 - 15164 y_9_1
      - 15164 y_9_2 - 15164 y_9_3 - 15164 y_9_4 - 15126 y_9_5 - 15125 y_21_0
      - 15164 y_21_1 - 15164 y_21_2 - 15164 y_21_3 - 15164 y_21_4 - 15125 y_21_5
      - 15113 y_12_0 - 15164 y_12_1 - 15164 y_12_2 - 15164 y_12_3 - 15164 y_12_4
      - 15113 y_12_5 + [ 15164 y_8_0^2 + 15164 y_8_0*y_8_1 + 15164 y_8_0*y_8_2
      + 15164 y_8_0*y_8_3 + 15164 y_8_0*y_8_4 + 15164 y_8_0*y_8_5
      + 15164 y_8_0*y_6_0 + 38 y_8_0*y_6_1 + 15164 y_8_0*y_2_0 + 46 y_8_0*y_2_1
      + 15164 y_8_0*y_9_0 + 80 y_8_0*y_9_1 + 15164 y_8_0*y_21_0
     